# **테슬라 10-K 보고서 RAG**

- 테슬라 10-K 보고서 데이터를 Neo4j 지식그래프에 저장
- 벡터 검색 기능을 활용한 RAG 시스템을 구축

---

## 1. 환경 설정

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

`(3) Neo4j 설정`

In [3]:
import os
from langchain_neo4j import Neo4jGraph

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

# LangChain 도구 활용 - DB 연결 객체 초기화 
graph = Neo4jGraph( 
    url=NEO4J_URI, 
    username=NEO4J_USERNAME, 
    password=NEO4J_PASSWORD,
    refresh_schema=True,
    )

graph.query("MATCH (n) RETURN n LIMIT 5;")

[]

`(4) 기존 DB의 모든 내용 삭제`

In [ ]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

In [ ]:
# 그래프 스키마 조회
graph.refresh_schema()
print(graph.schema)

## 2. 지식그래프 스키마 설계

* **주요 엔티티 (노드)**:

   1. **Document (문서)**: 
      - 지식 베이스의 기본 단위
      - 고유 ID로 식별되며 제목, 설명 및 생성/수정 타임스탬프 포함

   2. **Section (섹션)**:
      - 문서 내의 논리적 구분
      - 문서 내에서의 순서(order)와 제목 포함
      - (name, document_id)의 복합키로 고유하게 식별

   3. **Chunk (청크)**:
      - 실제 콘텐츠를 포함하는 최소 단위
      - 벡터 임베딩 저장 (차원: 1536) - embedding 속성
      - 토큰 수와 순서 정보 포함

* **관계**:

   1. **CONTAINS**: 상위 엔티티가 하위 엔티티를 포함하는 관계
      - Document → Section : `HAS_SECTION`
      - Section → Chunk : `FIRST_CHUNK`

   2. **RELATES_TO**: 청크 간의 의미적 관계
      - 관계 유형, 가중치 및 생성 시간 포함 : `NEXT`

* **제약조건**:

   1. Document의 id는 고유해야 함
   2. Section은 (name, document_id) 조합으로 고유하게 식별
   3. Chunk의 chunk_id는 고유해야 함

In [ ]:
# Document 노드 레이블 및 속성 정의 (제약조건 설정)

cypher_query = """
CREATE CONSTRAINT IF NOT EXISTS    // 제약조건 생성
FOR (d:Document)   // Document 레이블을 가진 노드에 대해
REQUIRE d.id IS UNIQUE;  // id 속성이 유일해야 함
"""

graph.query(cypher_query)

In [ ]:
# Section 노드 레이블 및 속성 정의 (제약조건 설정)

cypher_query = """
CREATE CONSTRAINT IF NOT EXISTS  // 제약조건 생성
FOR (s:Section)  // Section 레이블을 가진 노드에 대해
REQUIRE (s.name, s.document_id) IS NODE KEY; // name과 document_id 속성이 유일해야 함 (복합키)
"""

graph.query(cypher_query)

In [ ]:
# Chunk 노드 레이블 및 속성 정의 (제약조건 설정)

cypher_query = """
CREATE CONSTRAINT IF NOT EXISTS   // 제약조건 생성
FOR (c:Chunk)  // Chunk 레이블을 가진 노드에 대해
REQUIRE c.chunk_id IS UNIQUE;  // chunk_id 속성이 유일해야 함
"""

graph.query(cypher_query)

In [ ]:
# 벡터 인덱스 생성
cypher_query = """
CREATE VECTOR INDEX chunk_content_index IF NOT EXISTS
FOR (c:Chunk) 
ON (c.embedding)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }
}
"""

graph.query(cypher_query)

## 3. 데이터 구조화 및 저장

* **노드 구조**:
   - **Document 노드**: 전체 10-K 문서 표현
   - **Section 노드**: 문서의 각 섹션 표현 (Business, Risk Factors 등)
   - **Chunk 노드**: 분할된 텍스트 청크

* **관계 구조**:
   - `(:Document)-[:HAS_SECTION]->(:Section)`
   - `(:Section)-[:FIRST_CHUNK]->(:Chunk)`
   - `(:Chunk)-[:NEXT]->(:Chunk)` 


In [ ]:
import pickle

# 저장된 섹션 데이터 로드
with open("data/tesla_10k_sections_split.pkl", "rb") as f:
    section_docs_split = pickle.load(f)

# 섹션 데이터 확인
print(f"Number of sections: {len(section_docs_split)}")

In [ ]:
# 섹션 데이터 확인
section_docs_split.keys()

In [ ]:
# Business 섹션 데이터 확인
section_docs_split["Business"][0].to_json()


In [ ]:
section_docs_split["Business"][-1].to_json()

In [ ]:
section_docs_split["Business"][-1].metadata

In [ ]:
# 문서 ID 설정
doc_id = "tsla-20241231-gen"
print(f"Document ID: {doc_id}")

In [ ]:
# 문서 소스 경로 설정
doc_source = "data/tsla-20241231-gen.pdf"
print(f"Document Source: {doc_source}")

In [ ]:
import uuid
from langchain_openai import OpenAIEmbeddings

# OpenAI Embeddings 객체 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Document 노드 생성 함수
def create_document_node(graph, doc_id, source):
    """
    문서 노드를 생성하거나 업데이트합니다.
    """
    query = """
    MERGE (d:Document {id: $doc_id})
    SET d.source = $source
    RETURN d
    """
    return graph.query(
        query,
        params={
            "doc_id": doc_id,
            "source": source
        }
    )

# Section 노드 생성 함수 (section_number 추가)
def create_section_node(graph, section_name, section_number, doc_id):
    """
    섹션 노드를 생성하고 문서와 연결합니다.
    section_number (예: 'Item 1')를 포함합니다.
    """
    query = """
    MATCH (d:Document {id: $doc_id})
    MERGE (s:Section {
        name: $section_name, 
        section_number: $section_number,
        document_id: $doc_id
    })
    MERGE (d)-[:HAS_SECTION]->(s)
    RETURN s
    """
    return graph.query(
        query, 
        params={
            "section_name": section_name, 
            "section_number": section_number,
            "doc_id": doc_id
        }
    )

# Chunk 노드 생성 함수 
def create_chunk_node(graph, section_name, section_number, doc_id, chunk_id, content, embedding, metadata, is_first_in_section=False):
    """
    청크 노드를 생성하고 섹션과 연결합니다.
    metadata에서 page, order 정보를 사용합니다.
    is_first_in_section이 True인 경우 섹션과 직접 연결합니다.
    """
    # 임베딩 벡터가 numpy 배열인 경우 리스트로 변환
    if hasattr(embedding, "tolist"):  
        embedding = embedding.tolist()
    
    # 벡터 임베딩 생성 (db.create.setNodeVectorProperty 사용)
    query = """
    MATCH (s:Section {
        name: $section_name, 
        section_number: $section_number,
        document_id: $doc_id
    })
    CREATE (c:Chunk {
        chunk_id: $chunk_id,
        content: $content,
        order: $order,
        page: $page,
        section_name: $section_name,
        section_number: $section_number
    })
    WITH c, s
    CALL db.create.setNodeVectorProperty(c, 'embedding', $embedding)
    """
    
    # 각 섹션의 첫 번째 청크인 경우 섹션과 직접 연결
    if is_first_in_section:
        query += """
    CREATE (s)-[:FIRST_CHUNK]->(c)
    """
    
    query += """
    RETURN c
    """
    
    params = {
        "chunk_id": chunk_id,
        "content": content,
        "embedding": embedding,
        "order": metadata['order'],
        "page": metadata['page'],
        "section_name": section_name,
        "section_number": section_number,
        "doc_id": doc_id
    }
    
    return graph.query(query, params=params)

# 청크 간 순서 관계 생성 함수 (변경 없음)
def create_next_relationship(graph, prev_chunk_id, next_chunk_id):
    """
    두 청크 간의 순서 관계(NEXT)를 생성합니다.
    """
    query = """
    MATCH (prev:Chunk {chunk_id: $prev_chunk_id})
    MATCH (next:Chunk {chunk_id: $next_chunk_id})
    MERGE (prev)-[:NEXT]->(next)
    """
    return graph.query(
        query,
        params={
            "prev_chunk_id": prev_chunk_id,
            "next_chunk_id": next_chunk_id
        }
    )

# 전체 문서 처리 함수
def process_documents_to_graph(graph, documents, doc_id, source):
    """
    문서 리스트를 그래프로 변환합니다.
    
    Args:
        graph: Neo4j 그래프 객체
        documents: 처리할 문서 리스트
        doc_id: 문서 ID
        source: 문서 소스 (예: 파일명)
    """
    # 1. Document 노드 생성
    create_document_node(graph, doc_id, source)
    
    # 2. 섹션별로 그룹화
    sections = {}
    for doc in documents:
        section_name = doc.metadata.get('section', 'Unknown')
        section_number = doc.metadata.get('section_number', '')
        section_key = (section_name, section_number)
        
        if section_key not in sections:
            sections[section_key] = []
        sections[section_key].append(doc)
    
    # 3. 각 섹션 처리
    total_chunks = 0
    for (section_name, section_number), section_docs in sections.items():
        # Section 노드 생성
        create_section_node(graph, section_name, section_number, doc_id)
        
        # 섹션 내 문서들을 order로 정렬
        section_docs.sort(key=lambda x: (x.metadata.get('page', 0), x.metadata.get('order', 0)))
        
        # 청크 생성 및 연결
        prev_chunk_id = None
        for idx, doc in enumerate(section_docs):
            # 청크 ID 생성
            chunk_id = str(uuid.uuid4())
            
            # 임베딩 생성
            embedding = embeddings.embed_query(doc.page_content)
            
            # 각 섹션의 첫 번째 청크인지 확인
            is_first_in_section = (idx == 0)
            
            # Chunk 노드 생성
            create_chunk_node(
                graph, 
                section_name, 
                section_number,
                doc_id, 
                chunk_id, 
                doc.page_content, 
                embedding, 
                doc.metadata,
                is_first_in_section=is_first_in_section
            )
            
            # 이전 청크와 연결
            if prev_chunk_id:
                create_next_relationship(graph, prev_chunk_id, chunk_id)
            
            prev_chunk_id = chunk_id
            total_chunks += 1
    
    print(f"✅ 문서 '{source}'가 그래프에 성공적으로 추가되었습니다.")
    print(f"   - 섹션 수: {len(sections)}")
    print(f"   - 총 청크 수: {total_chunks}")
    
    # 섹션별 상세 정보 출력
    print("\n📊 섹션별 청크 분포:")
    for (section_name, section_number), section_docs in sorted(sections.items()):
        print(f"   - [{section_number}] {section_name}: {len(section_docs)} chunks")
    
# 문서 처리
process_documents_to_graph(
    graph=graph,
    documents=[doc for section in section_docs_split.values() for doc in section],
    doc_id=doc_id,
    source=doc_source
)

In [ ]:
# 청크 노드 수 확인
chunk_count_query = """
MATCH (c:Chunk)
RETURN COUNT(c) AS chunk_count
"""
# 청크 노드 수 쿼리 실행
graph.query(chunk_count_query)

In [ ]:
# 섹션 노드 수 확인
section_count_query = """
MATCH (s:Section)
RETURN COUNT(s) AS section_count
"""
# 섹션 노드 수 쿼리 실행
graph.query(section_count_query)

In [ ]:
# 특정 섹션의 모든 청크 조회
section_chunks_query = """
MATCH (s:Section {name: "Business", document_id: "tsla-20241231-gen"})
MATCH (s)-[:FIRST_CHUNK]->(first:Chunk)
MATCH path = (first)-[:NEXT*0..]->(c:Chunk)
RETURN c.content, c.page, c.order
ORDER BY c.page, c.order
"""
# 특정 섹션의 모든 청크 조회 쿼리 실행
graph.query(section_chunks_query)

In [ ]:
# 특정 페이지의 모든 청크 조회
page_chunks_query = """
MATCH (c:Chunk {page: 5})
RETURN c.section_name, c.content, c.order
ORDER BY c.order
"""
# 특정 페이지의 모든 청크 조회 쿼리 실행
graph.query(page_chunks_query)

In [ ]:
# 벡터 검색 (임베딩 유사도 기반)
vector_search_query = """
CALL db.index.vector.queryNodes('chunk_content_index', 10, $query_embedding)
YIELD node as chunk, score
RETURN chunk.content, chunk.section_name, chunk.page, score
ORDER BY score DESC
"""
# 쿼리 실행
query_embedding = embeddings.embed_query("Tesla's business overview")
vector_search_results = graph.query(
    vector_search_query,
    params={"query_embedding": query_embedding}
)
# 결과 출력
for result in vector_search_results:
    print(f"Section: {result['chunk.section_name']}, Page: {result['chunk.page']}, Score: {result['score']}")
    print(result['chunk.content'])
    print("-" * 80)

## 4. 벡터 검색 및 RAG 구현

In [ ]:
from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings

# OpenAI Embeddings 객체 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Neo4j 벡터 스토어 초기화 (기존 인덱스 사용)
vector_store = Neo4jVector.from_existing_index(
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME, 
    password=NEO4J_PASSWORD,
    index_name="chunk_content_index",
    node_label="Chunk",
    text_node_property="content",
    embedding_node_property="embedding",
    retrieval_query="""
    MATCH (c:Chunk {chunk_id: node.chunk_id}) // 청크 노드와 매칭
    
    // 이전 및 다음 청크 노드와 매칭
    OPTIONAL MATCH window = (prev:Chunk)-[:NEXT*0..1]->(c)-[:NEXT*0..1]->(next:Chunk)
    
    WITH 
      // window 경로에서 노드 추출
      apoc.text.join([chunk in nodes(window) | chunk.content], '\\n\\n') AS context_text,  // 청크 노드의 내용을 결합하여 문맥 텍스트 생성
      c.section_name AS section_name,   // 청크에 저장된 섹션 정보 사용
      c.section_number AS section_number, // 청크에 저장된 섹션 번호
      c.order AS chunk_order,            // 청크 노드의 순서
      c.chunk_id AS chunk_id,            // 청크 노드의 고유 ID
      score                               // 점수 정보 (유사도 점수)
    RETURN context_text AS text, score, {  
      section: section_name,
      item: section_number,
      chunk_order: chunk_order,
      chunk_id: chunk_id
    } AS metadata  // 메타데이터 생성
    """
)

In [ ]:
# 테스트 질문 (보고서 p.14에서 인용)
test_query = "What recognition did Tesla receive in the 2024 American Opportunity Index??"

retrieved_docs = vector_store.similarity_search_with_score(test_query, k=5)
print(f"검색된 문서 수: {len(retrieved_docs)}")

# 검색된 문서 및 점수 확인
for doc, score in retrieved_docs:
    print(f"Score: {score}")
    print(f"Section: {doc.metadata['section']}")
    print(f"Item: {doc.metadata['item']}")
    print(f"Chunk Order: {doc.metadata['chunk_order']}")
    print(f"Text: {doc.page_content[:50]}...")
    print("-" * 100)

In [ ]:
def remove_duplicates(results):
    """
    여러 기준으로 중복 제거
    """

    deduplicated = []
    seen_chunks = set()

    if isinstance(results[0], tuple):
        results = [result[0] for result in results]
    
    for doc in results:
        chunk_id = doc.metadata.get('chunk_id')
        # 청크 ID 기준 중복 제거
        if chunk_id and chunk_id in seen_chunks:
            continue
        seen_chunks.add(chunk_id)
        deduplicated.append(doc)
    
    return deduplicated 

print(f"검색된 문서 수: {len(retrieved_docs)}")
deduplicated_docs = remove_duplicates(retrieved_docs)
print(f"중복 제거 후 문서 수: {len(deduplicated_docs)}")


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# RAG용 프롬프트 템플릿
template = """
당신은 테슬라 10-K 보고서 전문가입니다. 다음 정보를 바탕으로 질문에 답변해 주세요. 
질문과 같은 언어로 답변해 주세요.

[10-K 보고서 내용]
{context}

[질문]
{question}

[답변] 
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# RAG 체인 구성
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
    )
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def format_retriever_results(results):
    # 검색된 결과에서 청크 내용을 추출하여 리스트로 반환
    results = remove_duplicates(results)
    return "\n\n".join([doc.page_content for doc in results])

# RAG 체인 구성
rag_chain = (
    {"context": retriever | format_retriever_results, "question": RunnablePassthrough()}
    | prompt 
    | llm 
    | StrOutputParser()
)

# 질문에 대한 답변 생성
test_query = "What recognition did Tesla receive in the 2024 American Opportunity Index?"
response = rag_chain.invoke(test_query)

print(response)

In [ ]:
# 질문에 대한 답변 생성
test_query = "테슬라의 2024년 사업 실적에 대해서 설명해 주세요."
response = rag_chain.invoke(test_query)

print(response)